# NephroTwin: build and validate the kidney digital twin

This notebook rebuilds every artefact in the repository from scratch:

1. Install dependencies and Java
2. Generate a synthetic patient population with Synthea
3. Run the test suite
4. Build the twin: data cleaning, population model, validation and figures
5. Inspect the results and package the artefacts

Runtime: 30 to 90 minutes, almost all of it in step 2. Keep this tab open while it runs.

## 1. Get the code and install dependencies

In [ ]:
REPO_URL = "https://github.com/<owner>/nephrotwin.git"   # replace <owner> with the GitHub account name

import os
if not os.path.exists("nephrotwin"):
    !git clone -q {REPO_URL}
%cd nephrotwin
!pip -q install -r requirements-dev.txt
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!java -version

## 2. Optional: keep generated data safe in Google Drive

Colab sessions can disconnect. Mounting Drive lets you copy the generated data there afterwards.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Generate the synthetic population

Synthea simulates complete lifetimes, so this is the slow step. 1,000 adults aged 40 to 85 typically take 30 to 90 minutes in Colab.
To go faster, run two smaller batches with different seeds; the pipeline combines them.

In [ ]:
!bash scripts/generate_synthea.sh 1000 42 data/raw/synthea_s42

In [ ]:
# Optional second batch (more patients, more progression events)
# !bash scripts/generate_synthea.sh 500 7 data/raw/synthea_s7

In [ ]:
if USE_DRIVE:
    !mkdir -p /content/drive/MyDrive/nephrotwin_data
    !cp -r data/raw/synthea_s* /content/drive/MyDrive/nephrotwin_data/
    print("Copied to Drive")

## 4. Run the tests

In [ ]:
!python -m pytest -q

## 5. Build the twin and the validation report

In [ ]:
import glob
batches = sorted(glob.glob("data/raw/synthea_s*/csv"))
print("Batches:", batches)
!python -m nephrotwin.pipeline --synthea {" ".join(batches)} --provenance "Synthea v4.0.0, Massachusetts, ages 40-85, FHIR export off" 

## 6. Inspect the results

In [ ]:
import json
m = json.load(open("reports/metrics.json"))
print(json.dumps(m["cohort"], indent=2))
v = m["validation"]
print("\nNext-result MAE:", {k: round(x["mae"], 2) for k, x in v["next_visit"].items()})
print("1 year+ MAE:    ", {k: round(x["mae"], 2) for k, x in v["one_year_plus"].items()})
print("\nPaired comparisons (negative favours the twin):")
for k, x in v["paired_comparisons"].items():
    print(f"  {k}: {x['mae_difference']:+.2f}  (95% CI {x['ci95_lower']:+.2f} to {x['ci95_upper']:+.2f})")
print("\nInterval coverage:", v["interval_calibration"])
print("Event prediction:", v["event_prediction"])

In [ ]:
from IPython.display import Image, display
for f in ["learning_curve", "method_comparison", "interval_calibration", "example_twin"]:
    display(Image(f"reports/figures/{f}.png"))

## 7. Package the artefacts for the repository

In [ ]:
!zip -qr nephrotwin_artifacts.zip data/processed models reports
from google.colab import files
files.download("nephrotwin_artifacts.zip")

Unzip the download and upload the `data/processed`, `models` and `reports` folders to the repository.
The dashboard reads these files.